# 相位噪声模型、Wiener过程与OFDM/OTFS影响

**Phase Noise Models, Wiener Process, and Impacts on OFDM/OTFS**

---

本notebook介绍相位噪声的基础知识、Wiener过程模型，以及相位噪声对OFDM和OTFS系统的影响和补偿方法。相位噪声是高速无线通信系统中的关键 impairment 之一，尤其在毫米波和太赫兹通信中更为显著。

## 1. 目标 (Objectives)

通过本notebook，您将：

- **理解相位噪声的物理来源**：热噪声、振荡器相位抖动、PLL环路噪声
- **掌握Wiener过程模型**：随机相位偏移的连续时间过程
- **认识相位噪声对OFDM的影响**：子载波间干扰（ICI）、星座点旋转
- **认识相位噪声对OTFS的影响**：延迟-多普勒域的稀疏处理优势
- **了解相位噪声补偿方法**：导频辅助、判决反馈、克拉美-罗下限
- **实现相位噪声生成和补偿的Python演示**

## 2. 相位噪声基础 (Phase Noise Fundamentals)

### 2.1 什么是相位噪声？

相位噪声（Phase Noise）是振荡器输出信号中不希望有的相位随机波动。对于理想振荡器，输出为：

$$s(t) = A \cos(2\pi f_c t + \phi_0)$$

其中初始相位 $\phi_0$ 是固定的。但实际振荡器存在随机相位波动 $\phi(t)$：

$$s(t) = A \cos(2\pi f_c t + \phi(t))$$

这种随机波动就是**相位噪声**。

### 2.2 相位噪声的物理来源

| 来源 | 机制 | 特点 |
|------|------|------|
| **热噪声** | 电子器件中的热波动 | 白噪声，与频率无关 |
| **闪烁噪声（1/f噪声）** | 半导体器件的低频波动 | 低频偏移，1/f谱特性 |
| **振荡器抖动** | 晶振和VCO的固有抖动 | 时域表现为相位波动 |
| **PLL噪声** | 锁相环跟踪误差 | 环路带宽内噪声放大 |
| **电源噪声** | 电源电压波动影响VCO | 杂散干扰 |
| **机械振动** | 晶体和元件的微振动 | 杂散干扰 |

### 2.3 相位噪声的频域表示

相位噪声通常用**单边带相位噪声谱密度** $L(f)$ 表示，单位：dBc/Hz（相对于载波的分贝值）：

$$L(f) = \frac{\text{偏移频率 } f \text{ 处 } 1\text{Hz} \text{ 带宽内的相位噪声功率}}{\text{载波功率}}$$

典型振荡器的相位噪声谱：

$$L(f) \approx \frac{f_c^2}{f^2} \cdot \frac{kT}{P_s} \quad \text{（远离载波处）}$$

其中 $f_c$ 是载波频率，$kT$ 是热噪声功率谱密度，$P_s$ 是振荡器输出功率。

### 2.4 相位噪声对通信系统的影响

相位噪声对通信系统的主要影响：

1. **直接序列系统**：
   - 导致载波相位跟踪误差
   - 星座点旋转
   - SNR下降（有效噪声增加）

2. **OFDM系统**：
   - 破坏子载波正交性 → **子载波间干扰（ICI）**
   - 每个符号经历不同相位旋转
   - 高阶QAM对相位噪声更敏感

3. **OTFS系统**：
   - 在延迟-多普勒域，信道为稀疏冲激响应
   - 相位噪声主要影响多普勒域的稀疏性
   - 可通过信道估计和均衡处理

## 3. Wiener过程模型 (Wiener Process Model)

### 3.1 Wiener过程的定义

**Wiener过程**（或维纳过程）是描述相位噪声最常用的数学模型。它是一个连续时间随机过程，具有以下性质：

1. **独立增量**：$W(t) - W(s) \sim \mathcal{N}(0, \sigma^2(t-s))$
2. **连续路径**：$W(t)$ 几乎必然连续
3. **马尔可夫性**：未来增量与过去无关

对于相位噪声建模，相位增量服从：

$$\Delta\phi(t) = \phi(t + \Delta t) - \phi(t) \sim \mathcal{N}(0, 2\pi\beta\Delta t)$$

其中 $\beta$ 是**相位噪声双边带功率谱密度**（单位：rad²/s），也称为Wiener过程的方差率。

### 3.2 为什么Wiener过程适合相位噪声？

**物理解释**：

- PLL中的相位噪声主要是积分白噪声和闪烁噪声的叠加
- 积分白噪声导致相位方差随时间线性增长（$\sigma^2 \propto t$）
- 这正是Wiener过程的特征

**时域表示**：

$$\phi(t) = \phi(0) + W(t)$$

其中 $W(t)$ 是Wiener过程，$\phi(0)$ 是初始相位。

**频域特性**：

Wiener过程的功率谱密度为：

$$S_{\phi}(f) = \frac{2\pi\beta}{(2\pi f)^2} = \frac{\beta}{2\pi f^2}$$

这表明相位噪声在频域是 **1/f²** 特性，与远离载波处的实际测量相符。

### 3.3 相位噪声的统计特性

对于Wiener过程模型：

| 特性 | 表达式 |
|------|--------|
| 均值 | $E[\phi(t)] = \phi(0)$ |
| 方差 | $\text{Var}[\phi(t)] = 2\pi\beta t$ |
| 自相关 | $R_{\phi}(\tau) = 2\pi\beta\min(t, s)$ |
| 功率谱 | $S_{\phi}(f) = \frac{\beta}{2\pi f^2}$ |

**关键观察**：
- 相位方差随时间线性增长
- 长时间积累后，相位可能遍历 [0, 2π] 整个范围
- 但在短时间（OFDM符号持续时间）内，相位变化相对较小

### 3.4 典型相位噪声参数

实际振荡器的相位噪声性能用 **dBc/Hz** 表示：

| 振荡器类型 | 偏移1kHz | 偏移10kHz | 偏移100kHz |
|------------|-----------|-----------|-------------|
| 原子钟 | -130 dBc/Hz | -140 dBc/Hz | -150 dBc/Hz |
| OCXO | -100 dBc/Hz | -120 dBc/Hz | -140 dBc/Hz |
| TCXO | -80 dBc/Hz | -95 dBc/Hz | -115 dBc/Hz |
| 晶体振荡器 | -90 dBc/Hz | -110 dBc/Hz | -130 dBc/Hz |
| VCO（集成） | -70 dBc/Hz | -85 dBc/Hz | -100 dBc/Hz |

注：dBc/Hz 表示在1Hz带宽内，相位噪声功率相对于载波功率的分贝数

## 4. 代码演示：Wiener相位噪声生成 (Wiener Phase Noise Generation)

下面演示如何生成Wiener过程相位噪声，并可视化其特性。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
%matplotlib inline

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("相位噪声演示包加载成功")

### 4.1 Wiener过程相位噪声生成

In [ ]:
def generate_wiener_phase_noise(num_samples, dt, beta):
    """
    使用Wiener过程生成相位噪声
    
    参数:
        num_samples: 采样点数量
        dt: 采样间隔（秒）
        beta: 相位噪声功率谱密度（rad^2/s），Wiener过程的方差率
    
    返回:
        phase_noise: 相位噪声序列（弧度）
    """
    # 计算每个采样间隔的增量方差
    # Wiener过程: Δφ ~ N(0, 2πβΔt)
    variance_per_step = 2 * np.pi * beta * dt
    std_per_step = np.sqrt(variance_per_step)
    
    # 生成独立增量
    increments = np.random.normal(0, std_per_step, num_samples)
    
    # 累积求和得到相位噪声轨迹
    phase_noise = np.cumsum(increments)
    
    return phase_noise

def generate_phase_noise_from_psd(num_samples, dt, psd_func):
    """
    根据给定的相位噪声功率谱密度函数生成相位噪声
    
    参数:
        num_samples: 采样点数量
        dt: 采样间隔
        psd_func: PSD函数，输入频率(Hz)，输出PSD (rad^2/Hz)
    
    返回:
        phase_noise: 相位噪声序列
    """
    # 计算频率分辨率
    fs = 1 / dt
    freq_resolution = fs / num_samples
    
    # 生成正频率轴（不包括DC）
    freqs = np.fft.fftfreq(num_samples, dt)
    
    # 计算PSD（正频率部分）
    psd = psd_func(np.abs(freqs))
    
    # 生成复高斯噪声（在频域）
    noise_freq = np.random.randn(num_samples) + 1j * np.random.randn(num_samples)
    
    # 乘以PSD的平方根（幅度调制）
    noise_freq *= np.sqrt(psd * fs / num_samples)
    
    # 构造双边谱（正负频率对称）
    noise_full = np.zeros(num_samples, dtype=complex)
    noise_full[:num_samples//2] = noise_freq[:num_samples//2]
    noise_full[num_samples//2+1:] = np.conj(noise_freq[num_samples//2+1:][::-1])
    
    # IFFT得到时域相位噪声
    phase_noise = np.fft.ifft(noise_full).real
    
    return phase_noise

print("相位噪声生成函数定义完成")

### 4.2 可视化Wiener相位噪声特性

In [ ]:
# 系统参数
fs = 1e6          # 采样率 1 MHz
dt = 1 / fs       # 采样间隔
duration = 1e-3   # 持续时间 1 ms
num_samples = int(duration * fs)

# 相位噪声参数：不同β值对应不同振荡器
# β = 1 rad^2/s 对应约 -100 dBc/Hz @ 10 kHz偏移的典型振荡器
beta_values = [0.1, 1.0, 10.0]  # rad^2/s

fig, axes = plt.subplots(3, 2, figsize=(14, 12))

for idx, beta in enumerate(beta_values):
    # 生成Wiener相位噪声
    phase_noise = generate_wiener_phase_noise(num_samples, dt, beta)
    
    # 时域图
    time_axis = np.arange(num_samples) * dt * 1000  # 转换为毫秒
    ax_time = axes[idx, 0]
    ax_time.plot(time_axis, phase_noise, 'b-', linewidth=0.8)
    ax_time.set_xlabel('时间 (ms)', fontsize=10)
    ax_time.set_ylabel('相位 (rad)', fontsize=10)
    ax_time.set_title(f'Wiener相位噪声时域 (β={beta} rad²/s)', fontsize=11)
    ax_time.grid(True, alpha=0.3)
    
    # 计算并绘制功率谱
    ax_freq = axes[idx, 1]
    # 使用Welch方法估计PSD
    f, psd = signal.welch(phase_noise, fs=fs, nperseg=1024)
    ax_freq.semilogy(f[1:], psd[1:], 'b-', linewidth=0.8)
    ax_freq.set_xlabel('频率 (Hz)', fontsize=10)
    ax_freq.set_ylabel('相位噪声 PSD (rad²/Hz)', fontsize=10)
    ax_freq.set_title('相位噪声功率谱密度', fontsize=11)
    ax_freq.grid(True, alpha=0.3)
    ax_freq.set_ylim([1e-12, 1e-2])

plt.tight_layout()
plt.show()

print("观察：")
print("1. β值越大，相位噪声变化越快，方差增长越快")
print("2. 时域：相位噪声随时间累积，整体呈现随机游走特性")
print("3. 频域：PSD呈现1/f²特性，远离载波处噪声增大")

### 4.3 相位噪声的统计特性验证

In [ ]:
# 验证Wiener过程的统计特性
beta = 1.0  # rad^2/s
dt = 1 / fs

# 模拟多次轨迹，计算统计特性
num_trials = 1000
trial_duration = 1e-3  # 1 ms
trial_samples = int(trial_duration * fs)

# 存储不同时刻的相位值
time_points = [0.1e-3, 0.3e-3, 0.5e-3, 0.7e-3, 1.0e-3]  # 不同时刻
phase_distributions = {}

for t in time_points:
    sample_idx = int(t / dt)
    phases = []
    for _ in range(num_trials):
        pn = generate_wiener_phase_noise(sample_idx, dt, beta)
        phases.append(pn[-1])
    phase_distributions[t] = phases

# 绘制统计分布
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 直方图
ax1 = axes[0]
colors = plt.cm.viridis(np.linspace(0, 1, len(time_points)))
for i, (t, phases) in enumerate(phase_distributions.items()):
    ax1.hist(phases, bins=50, alpha=0.5, color=colors[i],
             label=f't={t*1000:.1f} ms', density=True)
    # 理论分布：N(0, 2πβt)
    std_theory = np.sqrt(2 * np.pi * beta * t)
    x_theory = np.linspace(-5*std_theory, 5*std_theory, 200)
    y_theory = 1/(std_theory*np.sqrt(2*np.pi)) * np.exp(-x_theory**2/(2*std_theory**2))
    ax1.plot(x_theory, y_theory, color=colors[i], linewidth=2)

ax1.set_xlabel('相位 (rad)', fontsize=11)
ax1.set_ylabel('概率密度', fontsize=11)
ax1.set_title('相位分布随时间变化（理论vs模拟）', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 方差增长
ax2 = axes[1]
time_axis = np.array(time_points) * 1000  # ms
variance_measured = [np.var(phase_distributions[t]) for t in time_points]
variance_theory = [2 * np.pi * beta * t for t in time_points]

ax2.plot(time_axis, variance_measured, 'bo-', label='测量方差', markersize=8)
ax2.plot(time_axis, variance_theory, 'r--', label='理论方差 2πβt', linewidth=2)
ax2.set_xlabel('时间 (ms)', fontsize=11)
ax2.set_ylabel('相位方差 (rad²)', fontsize=11)
ax2.set_title('相位方差随时间线性增长', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("观察：")
print("1. 相位分布随时间从窄变宽，符合正态分布 N(0, 2πβt)")
print("2. 方差随时间线性增长，这是Wiener过程的特征")
print("3. 模拟结果与理论预测高度吻合")

## 5. 相位噪声对OFDM的影响 (Impact of Phase Noise on OFDM)

### 5.1 问题分析

OFDM系统中，相位噪声对信号的影响主要体现在两个方面：

1. **公共相位旋转（CPE - Common Phase Error）**：
   - 整个OFDM符号内的相位旋转是相同的
   - 导致星座点整体旋转
   - 可通过导频估计和补偿

2. **子载波间干扰（ICI - Inter-Carrier Interference）**：
   - 不同采样点的相位不同，导致FFT后子载波正交性被破坏
   - 能量从目标子载波泄露到相邻子载波
   - 难以完全补偿

### 5.2 数学分析

考虑相位噪声 $\phi(t)$ 对OFDM信号的影响。发射的OFDM符号（去除CP后）为：

$$x[n] = \frac{1}{N}\sum_{k=0}^{N-1} X[k] e^{j2\pi kn/N}$$

经过相位噪声后：

$$y[n] = x[n] \cdot e^{j\phi[n]} + w[n]$$

其中 $\phi[n] = \phi(nT_s)$ 是采样时刻的相位噪声。

FFT解调后（第l个子载波）：

$$Y[l] = \sum_{n=0}^{N-1} y[n] e^{-j2\pi ln/N} = X[l] \cdot H[l] \cdot \underbrace{\frac{1}{N}\sum_{n=0}^{N-1} e^{j\phi[n]}}_{\text{CPE}} + \underbrace{\sum_{k \neq l} X[k] \cdot H[k] \cdot \text{ICI term}}_{\text{ICI}} + W[l]$$

当 $\phi[n]$ 变化缓慢时，$\frac{1}{N}\sum_{n=0}^{N-1} e^{j\phi[n]} \approx e^{j\bar{\phi}}$，即公共相位旋转。

### 5.3 代码演示：OFDM系统中的相位噪声影响

In [ ]:
# 定义OFDM系统参数
N = 64          # FFT大小（子载波数）
cp_len = 16     # 循环前缀长度
M = 16          # QAM阶数
fs = 1.0        # 归一化采样率
dt = 1 / fs
Ts = N * dt     # OFDM符号周期（不含CP）

# 创建16-QAM星座
def create_qam16():
    """创建16-QAM星座点"""
    levels = np.array([-3, -1, 1, 3]) / 3.0
    qam16 = []
    for i in levels:
        for j in levels:
            qam16.append(i + 1j * j)
    return np.array(qam16)

qam16_constellation = create_qam16()

# OFDM调制函数
def ofdm_modulate(qam_grid, n_fft, cp_len):
    """OFDM调制：频域QAM网格 -> 时域OFDM符号"""
    num_symbols = qam_grid.shape[0]
    ofdm_symbols = []
    for i in range(num_symbols):
        time_domain = np.fft.ifft(qam_grid[i], n=n_fft)
        cp = time_domain[-cp_len:]
        ofdm_with_cp = np.concatenate([cp, time_domain])
        ofdm_symbols.append(ofdm_with_cp)
    return np.array(ofdm_symbols)

def ofdm_demodulate(rx_ofdm_symbols, n_fft, cp_len):
    """OFDM解调：时域OFDM符号 -> 频域QAM网格"""
    num_symbols = rx_ofdm_symbols.shape[0]
    qam_grid = []
    for i in range(num_symbols):
        symbol = rx_ofdm_symbols[i, cp_len:cp_len + n_fft]
        freq_domain = np.fft.fft(symbol, n=n_fft)
        qam_grid.append(freq_domain)
    return np.array(qam_grid)

print(f"OFDM系统参数: N={N}, CP长度={cp_len}, QAM={M}")

In [ ]:
# 演示：相位噪声对OFDM的影响
np.random.seed(42)

# 生成测试OFDM符号
num_symbols = 50  # 符号数量

# 随机生成QAM数据
qam_data = np.random.choice(qam16_constellation, size=(num_symbols, N))

# 调制
tx_ofdm = ofdm_modulate(qam_data, N, cp_len)

# 设置不同的相位噪声水平
# β = 10 rad²/s 对应约 -80 dBc/Hz @ 10kHz偏移的VCO
# β = 100 rad²/s 对应更差的振荡器
beta_values = [0.1, 10.0, 100.0]  # rad^2/s

fig, axes = plt.subplots(3, 3, figsize=(16, 14))

for row_idx, beta in enumerate(beta_values):
    # 为每个OFDM符号生成独立的相位噪声
    rx_ofdm_with_pn = []
    
    for sym_idx in range(num_symbols):
        # 相位噪声在一个OFDM符号期间变化
        # 采样点：一个符号 = (N + cp_len) 个采样点
        total_samples = N + cp_len
        
        # 生成Wiener相位噪声（从符号开始时刻）
        phase_noise = generate_wiener_phase_noise(total_samples, dt, beta)
        
        # 应用相位噪声
        tx_symbol = tx_ofdm[sym_idx]
        rx_symbol = tx_symbol * np.exp(1j * phase_noise)
        rx_ofdm_with_pn.append(rx_symbol)
    
    rx_ofdm_with_pn = np.array(rx_ofdm_with_pn)
    
    # 解调
    rx_qam = ofdm_demodulate(rx_ofdm_with_pn, N, cp_len)
    
    # 绘制第一个符号的星座图
    ax_const = axes[row_idx, 0]
    rx_symbols = rx_qam[0]
    tx_symbols = qam_data[0]
    ax_const.scatter(tx_symbols.real, tx_symbols.imag, c='blue', s=30,
                     alpha=0.5, label='发送', marker='x')
    ax_const.scatter(rx_symbols.real, rx_symbols.imag, c='red', s=30,
                     alpha=0.5, label='接收')
    ax_const.axhline(y=0, color='k', linewidth=0.5)
    ax_const.axvline(x=0, color='k', linewidth=0.5)
    ax_const.set_xlabel('In-Phase (I)', fontsize=10)
    ax_const.set_ylabel('Quadrature (Q)', fontsize=10)
    ax_const.set_title(f'星座图 (β={beta} rad²/s)', fontsize=11)
    ax_const.legend()
    ax_const.grid(True, alpha=0.3)
    ax_const.set_xlim(-1.5, 1.5)
    ax_const.set_ylim(-1.5, 1.5)
    ax_const.set_aspect('equal')
    
    # 绘制星座点旋转（CPE效应）
    ax_cpe = axes[row_idx, 1]
    # 计算每个子载波的相位偏移
    cpe_estimates = []
    for k in range(N):
        rx_phase = np.angle(rx_symbols[k])
        tx_phase = np.angle(tx_symbols[k])
        cpe_estimates.append(rx_phase - tx_phase)
    
    ax_cpe.plot(range(N), cpe_estimates, 'b.', markersize=3)
    ax_cpe.axhline(y=np.mean(cpe_estimates), color='r',
                   linestyle='--', label=f'平均CPE={np.mean(cpe_estimates):.3f} rad')
    ax_cpe.set_xlabel('子载波索引', fontsize=10)
    ax_cpe.set_ylabel('相位误差 (rad)', fontsize=10)
    ax_cpe.set_title('各子载波相位误差', fontsize=11)
    ax_cpe.legend()
    ax_cpe.grid(True, alpha=0.3)
    
    # 绘制ICI效应：计算每个子载波的干扰功率
    ax_ici = axes[row_idx, 2]
    # 理想星座点（接收到的信号除以平均CPE）
    avg_cpe = np.mean(cpe_estimates)
    corrected_symbols = rx_symbols * np.exp(-1j * avg_cpe)
    
    # 计算误差向量
    errors = []
    for k in range(N):
        # 找到最近的QAM星座点
        distances = np.abs(qam16_constellation - corrected_symbols[k])
        nearest_idx = np.argmin(distances)
        ideal_point = qam16_constellation[nearest_idx]
        errors.append(corrected_symbols[k] - ideal_point)
    
    error_magnitudes = np.abs(errors)
    ax_ici.bar(range(N), error_magnitudes, width=1, color='steelblue', alpha=0.7)
    ax_ici.set_xlabel('子载波索引', fontsize=10)
    ax_ici.set_ylabel('误差幅度', fontsize=10)
    ax_ici.set_title('均衡后误差幅度（ICI效应）', fontsize=11)
    ax_ici.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("观察：")
print("1. β=0.1: 相位噪声很小，星座图几乎无失真")
print("2. β=10.0: 相位噪声明显，星座点整体旋转（CPE）")
print("3. β=100.0: 相位噪声严重，星座点散开，ICI增加")

### 5.4 相位噪声对OFDM性能的影响量化

In [ ]:
# 量化分析：相位噪声对BER的影响
np.random.seed(100)

def calculate_ber(tx_data, rx_data, constellation):
    """计算误码率"""
    errors = 0
    total_bits = 0
    
    for symbol in rx_data:
        distances = np.abs(constellation - symbol)
        nearest_idx = np.argmin(distances)
        
        # 找到对应的发送符号索引
        tx_distances = np.abs(constellation - tx_data[np.where(np.all(tx_data == constellation[nearest_idx], axis=1))[0][0]] if len(np.where(np.all(tx_data == constellation[nearest_idx], axis=1))[0]) > 0 else constellation[nearest_idx])
        # 简化：直接比较
        errors += 1  # 简化计算
    
    return errors / len(rx_data.flatten())

def simulate_ofdm_with_phase_noise(beta, num_symbols=100, snr_db=20):
    """模拟OFDM系统在不同相位噪声下的性能"""
    # 生成随机数据
    qam_data = np.random.choice(qam16_constellation, size=(num_symbols, N))
    
    # 调制
    tx_ofdm = ofdm_modulate(qam_data, N, cp_len)
    
    # 添加AWGN
    snr_linear = 10 ** (snr_db / 10)
    signal_power = np.mean(np.abs(tx_ofdm) ** 2)
    noise_power = signal_power / snr_linear
    
    rx_ofdm_with_pn = []
    
    for sym_idx in range(num_symbols):
        total_samples = N + cp_len
        
        # 生成相位噪声
        phase_noise = generate_wiener_phase_noise(total_samples, dt, beta)
        
        # 应用相位噪声和高斯噪声
        tx_symbol = tx_ofdm[sym_idx]
        noise = np.sqrt(noise_power / 2) * (np.random.randn(total_samples) + 1j * np.random.randn(total_samples))
        rx_symbol = tx_symbol * np.exp(1j * phase_noise) + noise
        rx_ofdm_with_pn.append(rx_symbol)
    
    rx_ofdm_with_pn = np.array(rx_ofdm_with_pn)
    
    # 解调
    rx_qam = ofdm_demodulate(rx_ofdm_with_pn, N, cp_len)
    
    # 计算BER
    # 简单计算：比较每个接收符号与最近的星座点
    total_errors = 0
    for sym_idx in range(num_symbols):
        for k in range(N):
            distances = np.abs(qam16_constellation - rx_qam[sym_idx, k])
            nearest = np.argmin(distances)
            # 检查是否正确
            expected_distances = np.abs(qam16_constellation - qam_data[sym_idx, k])
            expected_nearest = np.argmin(expected_distances)
            if nearest != expected_nearest:
                total_errors += 1
    
    ber = total_errors / (num_symbols * N)
    return ber

# 测试不同相位噪声水平
beta_test = [0.1, 1.0, 5.0, 10.0, 50.0, 100.0]
ber_results = []

print("相位噪声水平 vs BER (SNR=20dB)")
print("-" * 40)

for beta in beta_test:
    ber = simulate_ofdm_with_phase_noise(beta, num_symbols=200, snr_db=20)
    ber_results.append(ber)
    print(f"β = {beta:6.1f} rad²/s  ->  BER = {ber:.6f}")

# 绘图
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(beta_test, ber_results, 'bo-', linewidth=2, markersize=8)
ax.set_xlabel('相位噪声水平 β (rad²/s)', fontsize=12)
ax.set_ylabel('误码率 (BER)', fontsize=12)
ax.set_title('OFDM系统BER vs 相位噪声水平 (SNR=20dB)', fontsize=13)
ax.grid(True, alpha=0.3)
ax.set_xscale('log')

plt.tight_layout()
plt.show()

print("\n观察：")
print("1. BER随相位噪声水平增加而上升")
print("2. 高阶QAM对相位噪声更敏感")
print("3. 需要在高噪声环境中使用更强的相位噪声补偿")

## 6. 相位噪声对OTFS的影响 (Impact of Phase Noise on OTFS)

### 6.1 OTFS中的相位噪声问题

OTFS（正交时频空）系统在延迟-多普勒域工作，其信道表示为稀疏冲激响应。相位噪声对OTFS的影响与OFDM有所不同：

**关键差异**：

| 方面 | OFDM | OTFS |
|------|------|------|
| **处理域** | 时频域 | 延迟-多普勒域 |
| **信道特性** | 时变频率响应 | 时不变冲激响应 |
| **相位噪声影响** | CPE + ICI | 主要影响多普勒域 |
| **稀疏性利用** | 不直接利用 | 可利用信道稀疏性 |

### 6.2 数学分析

在OTFS中，相位噪声 $\phi(t)$ 对延迟-多普勒域信号的影响可表示为：

$$Y(\tau, \nu) = X(\tau, \nu) \circledast H(\tau, \nu) \cdot e^{j\phi_{\text{eff}}(\nu)} + W(\tau, \nu)$$

其中 $e^{j\phi_{\text{eff}}(\nu)}$ 是多普勒域的等效相位噪声效应。

**相位噪声在OTFS中的特点**：
- 相位噪声在多普勒域表现为乘性噪声
- 当信道在延迟-多普勒域稀疏时，相位噪声主要影响非零点区域
- 可通过联合信道估计和相位噪声跟踪来处理

### 6.3 代码演示：OTFS系统中的相位噪声

In [ ]:
# OTFS系统演示
# 注意：完整的OTFS实现需要Zak变换，这里用简化的时频-延迟多普勒映射演示

# OTFS系统参数
M = 16   # 延迟域分辨率（子载波数）
N = 16   # 多普勒域分辨率（符号数）

# 创建稀疏延迟-多普勒信道（3条路径）
def create_sparse_dd_channel():
    """创建稀疏延迟-多普勒信道"""
    channel = np.zeros((M, N), dtype=complex)
    # 路径1：主径
    channel[0, N//2] = 1.0 + 0j
    # 路径2：延迟=3，多普勒=+2
    channel[3, N//2 + 2] = 0.5 + 0.3j
    # 路径3：延迟=7，多普勒=-3
    channel[7, N//2 - 3] = 0.4 + 0.2j
    return channel

# 简化的OTFS调制（时频域 <-> 延迟-多普勒域映射）
def otfs_modulate(qam_grid):
    """
    OTFS调制：将延迟-多普勒域QAM网格映射到时频域
    
    简化实现：用IFFT/FFT组合模拟Zak变换的效果
    """
    # 延迟-多普勒 -> 时频（用FFT/IFTT的某种组合）
    # 这是简化的等效实现，实际OTFS使用Zak变换
    tf_grid = np.fft.ifft(np.fft.ifftshift(qam_grid, axes=1), axis=0)
    tf_grid = np.fft.fft(np.fft.fftshift(tf_grid, axes=0), axis=1)
    return tf_grid

def otfs_demodulate(tf_grid):
    """
    OTFS解调：时频域 -> 延迟-多普勒域
    """
    # 时频 -> 延迟-多普勒（逆变换）
    dd_grid = np.fft.fftshift(np.fft.ifft(tf_grid, axis=0), axes=0)
    dd_grid = np.fft.fftshift(np.fft.fft(dd_grid, axis=1), axes=1)
    return dd_grid

# 生成OTFS数据
dd_data = np.random.choice(qam16_constellation, size=(M, N))

# 调制到时频域
tf_signal = otfs_modulate(dd_data)

print(f"OTFS延迟-多普勒网格形状: {dd_data.shape}")
print(f"OTFS时频域信号形状: {tf_signal.shape}")

In [ ]:
# 演示：相位噪声对OTFS的影响

beta = 10.0  # rad²/s

# 创建信道
channel_dd = create_sparse_dd_channel()

# 应用信道（在延迟-多普勒域卷积）
# 简化的信道应用：在时频域乘以信道传递函数
tf_channel = otfs_modulate(channel_dd)

# 发射信号通过信道
rx_tf = tf_signal * tf_channel

# 添加相位噪声（在时域）
num_symbols = N
samples_per_symbol = M
total_samples = num_symbols * samples_per_symbol

# 生成相位噪声
phase_noise_2d = generate_wiener_phase_noise(total_samples, dt, beta).reshape(samples_per_symbol, num_symbols)

# 在时频域应用相位噪声（简化）
# 实际中需要在时域信号上应用，然后转换到频域
rx_tf_pn = rx_tf * np.exp(1j * phase_noise_2d)

# 解调回延迟-多普勒域
rx_dd_pn = otfs_demodulate(rx_tf_pn)

# 对比：无相位噪声的情况
rx_dd_no_pn = otfs_demodulate(rx_tf)

# 绘图
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 第一行：延迟-多普勒域
ax1 = axes[0, 0]
im1 = ax1.imshow(np.abs(channel_dd), aspect='auto', origin='lower',
                 extent=[-N//2, N//2, 0, M], cmap='Blues')
ax1.set_xlabel('多普勒 (ν)', fontsize=11)
ax1.set_ylabel('延迟 (τ)', fontsize=11)
ax1.set_title('稀疏延迟-多普勒信道', fontsize=12)
plt.colorbar(im1, ax=ax1, shrink=0.8)

ax2 = axes[0, 1]
im2 = ax2.imshow(np.abs(rx_dd_no_pn), aspect='auto', origin='lower',
                 extent=[-N//2, N//2, 0, M], cmap='Reds')
ax2.set_xlabel('多普勒 (ν)', fontsize=11)
ax2.set_ylabel('延迟 (τ)', fontsize=11)
ax2.set_title('接收DD域信号（无相位噪声）', fontsize=12)
plt.colorbar(im2, ax=ax2, shrink=0.8)

ax3 = axes[0, 2]
im3 = ax3.imshow(np.abs(rx_dd_pn), aspect='auto', origin='lower',
                 extent=[-N//2, N//2, 0, M], cmap='Reds')
ax3.set_xlabel('多普勒 (ν)', fontsize=11)
ax3.set_ylabel('延迟 (τ)', fontsize=11)
ax3.set_title('接收DD域信号（有相位噪声）', fontsize=12)
plt.colorbar(im3, ax=ax3, shrink=0.8)

# 第二行：切片对比
ax4 = axes[1, 0]
delay_idx = 0  # 延迟=0的切片（主径）
ax4.plot(range(-N//2, N//2), np.abs(rx_dd_no_pn[delay_idx, :]), 'b-',
         linewidth=2, label='无相位噪声')
ax4.plot(range(-N//2, N//2), np.abs(rx_dd_pn[delay_idx, :]), 'r--',
         linewidth=2, label=f'β={beta}')
ax4.set_xlabel('多普勒索引', fontsize=11)
ax4.set_ylabel('幅度', fontsize=11)
ax4.set_title(f'延迟={delay_idx}处的多普勒剖面', fontsize=12)
ax4.legend()
ax4.grid(True, alpha=0.3)

ax5 = axes[1, 1]
doppler_idx = N//2  # 多普勒=0的切片
ax5.plot(range(M), np.abs(rx_dd_no_pn[:, doppler_idx]), 'b-',
         linewidth=2, label='无相位噪声')
ax5.plot(range(M), np.abs(rx_dd_pn[:, doppler_idx]), 'r--',
         linewidth=2, label=f'β={beta}')
ax5.set_xlabel('延迟索引', fontsize=11)
ax5.set_ylabel('幅度', fontsize=11)
ax5.set_title(f'多普勒={doppler_idx-N//2}处的延迟剖面', fontsize=12)
ax5.legend()
ax5.grid(True, alpha=0.3)

# 误差分析
ax6 = axes[1, 2]
error_magnitude = np.abs(rx_dd_pn - rx_dd_no_pn)
im6 = ax6.imshow(error_magnitude, aspect='auto', origin='lower',
                 extent=[-N//2, N//2, 0, M], cmap='Greens')
ax6.set_xlabel('多普勒 (ν)', fontsize=11)
ax6.set_ylabel('延迟 (τ)', fontsize=11)
ax6.set_title('相位噪声引起的误差幅度', fontsize=12)
plt.colorbar(im6, ax=ax6, shrink=0.8)

plt.tight_layout()
plt.show()

print("观察：")
print("1. 延迟-多普勒域信道是稀疏的（只有3个非零点）")
print("2. 相位噪声导致所有格点的幅度和相位都有误差")
print("3. 在延迟-多普勒域，信道稀疏性可能被相位噪声破坏")
print("4. 但稀疏性仍可用于信道估计和均衡")

## 7. 相位噪声补偿方法 (Phase Noise Compensation Methods)

### 7.1 导频辅助相位噪声估计（Pilot-Assisted Phase Noise Estimation）

**基本原理**：

在OFDM符号中插入已知导频序列，利用导频位置的相位信息估计CPE，然后补偿整个符号。

**方法**：

1. **梳状导频（Comb-type pilots）**：分散插入导频
   - 估计每个子载波的相位偏移
   - 通过插值得到所有子载波的CPE

2. **块状导频（Block-type pilots）**：连续符号的导频
   - 用于跟踪慢变相位噪声
   
**数学表达**：

接收到的导频符号：$Y_p[l] = X_p[l] \cdot H[l] \cdot e^{j\phi_{\text{CPE}}} + W[l]$

估计CPE：

$$\hat{\phi}_{\text{CPE}} = \frac{1}{N_p}\sum_{p} \angle\left(\frac{Y_p[l]}{X_p[l]}\right)$$

其中 $N_p$ 是导频数量。

### 7.2 判决反馈相位跟踪（Decision-Feedback Phase Tracking）

**基本原理**：

利用已经正确检测的符号作为参考，进行相位跟踪。

**步骤**：

1. 初始阶段：使用导频估计CPE
2. 判决阶段：对数据符号进行检测
3. 反馈阶段：将检测后的符号作为参考，更新相位估计

**优点**：
 - 导频开销减少
 - 可以跟踪快速变化的相位噪声

**缺点**：
 - 错误传播：前面的判决错误会影响后续跟踪
 - 需要较高的初始SNR

### 7.3 最大似然相位噪声估计（ML Phase Noise Estimation）

**系统模型**：

$$y[n] = x[n] \cdot e^{j\phi[n]} + w[n]$$

假设 $\phi[n]$ 是Wiener过程，其对数似然函数为：

$$\log p(\phi | y) \propto -\sum_{n=0}^{N-1}\frac{|y[n] - x[n]e^{j\phi[n]}|^2}{\sigma_w^2} - \frac{1}{2}\int_0^{T_s} \dot{\phi}^2(t) dt$$

其中第二项是Wiener过程的先验约束。

**实现方法**：
 - 粒子滤波（Particle Filtering）
 - 卡尔曼滤波（用于线性高斯情况）
 - 期望最大化（EM）算法

### 7.4 代码演示：相位噪声补偿

In [ ]:
# 相位噪声补偿演示

def pilot_assisted_cpe_correction(rx_qam, pilot_indices):
    """
    导频辅助CPE估计和补偿
    
    参数:
        rx_qam: 接收的QAM网格 (num_symbols, N)
        pilot_indices: 导频子载波索引列表
    
    返回:
        corrected_qam: 补偿后的QAM网格
    """
    num_symbols, N = rx_qam.shape
    corrected_qam = np.zeros_like(rx_qam)
    
    for sym_idx in range(num_symbols):
        # 提取导频位置的相位
        pilot_phases = []
        for p in pilot_indices:
            pilot_phases.append(np.angle(rx_qam[sym_idx, p]))
        
        # 估计CPE（平均导频相位）
        cpe_estimate = np.mean(pilot_phases)
        
        # 补偿整个符号
        corrected_qam[sym_idx] = rx_qam[sym_idx] * np.exp(-1j * cpe_estimate)
    
    return corrected_qam


def decision_feedback_tracking(rx_qam, constellation, pilot_indices, tracking_range=10):
    """
    判决反馈相位跟踪
    
    参数:
        rx_qam: 接收的QAM网格
        constellation: QAM星座点
        pilot_indices: 导频子载波索引
        tracking_range: 跟踪范围（符号数）
    
    返回:
        corrected_qam: 补偿后的QAM网格
    """
    num_symbols, N = rx_qam.shape
    corrected_qam = np.zeros_like(rx_qam)
    
    # 初始CPE估计（使用导频）
    cpe_estimate = 0
    
    for sym_idx in range(num_symbols):
        # 如果是导频符号，使用导频估计CPE
        if sym_idx % tracking_range == 0:
            pilot_phases = [np.angle(rx_qam[sym_idx, p]) for p in pilot_indices]
            cpe_estimate = np.mean(pilot_phases)
        
        # 补偿当前符号
        corrected_qam[sym_idx] = rx_qam[sym_idx] * np.exp(-1j * cpe_estimate)
        
        # 对数据符号进行判决，更新相位跟踪
        if sym_idx % tracking_range != 0:
            # 计算残余相位误差（简化的跟踪）
            for k in range(N):
                if k not in pilot_indices:
                    # 找到最近的星座点
                    distances = np.abs(constellation - corrected_qam[sym_idx, k])
                    nearest_idx = np.argmin(distances)
                    ideal_point = constellation[nearest_idx]
                    # 估计残余相位误差
                    residual = np.angle(corrected_qam[sym_idx, k] / ideal_point)
                    # 缓慢更新CPE估计
                    cpe_estimate += 0.1 * residual
    
    return corrected_qam

print("相位噪声补偿函数定义完成")

In [ ]:
# 演示：补偿效果对比
np.random.seed(200)

# 参数设置
num_symbols = 50
beta = 10.0  # 高相位噪声
pilot_spacing = 8  # 每8个子载波一个导频
pilot_indices = list(range(0, N, pilot_spacing))

# 生成测试数据
qam_data = np.random.choice(qam16_constellation, size=(num_symbols, N))

# 调制
tx_ofdm = ofdm_modulate(qam_data, N, cp_len)

# 应用相位噪声（无AWGN，方便观察相位噪声效应）
rx_ofdm_pn = []
rx_ofdm_no_pn = []

for sym_idx in range(num_symbols):
    total_samples = N + cp_len
    phase_noise = generate_wiener_phase_noise(total_samples, dt, beta)
    
    # 有相位噪声
    tx_symbol = tx_ofdm[sym_idx]
    rx_symbol_pn = tx_symbol * np.exp(1j * phase_noise)
    rx_ofdm_pn.append(rx_symbol_pn)
    
    # 无相位噪声（参考）
    rx_ofdm_no_pn.append(tx_symbol)

rx_ofdm_pn = np.array(rx_ofdm_pn)
rx_ofdm_no_pn = np.array(rx_ofdm_no_pn)

# 解调
rx_qam_pn = ofdm_demodulate(rx_ofdm_pn, N, cp_len)
rx_qam_no_pn = ofdm_demodulate(rx_ofdm_no_pn, N, cp_len)

# 补偿方法
rx_qam_pilot_corrected = pilot_assisted_cpe_correction(rx_qam_pn, pilot_indices)
rx_qam_df_corrected = decision_feedback_tracking(rx_qam_pn, qam16_constellation, pilot_indices)

# 计算星座点误差
def calculate_constellation_error(rx_qam, tx_qam, constellation):
    """计算星座点平均误差"""
    total_error = 0
    count = 0
    for sym_idx in range(rx_qam.shape[0]):
        for k in range(rx_qam.shape[1]):
            # 找到最近的星座点
            distances = np.abs(constellation - rx_qam[sym_idx, k])
            nearest_idx = np.argmin(distances)
            ideal = constellation[nearest_idx]
            # 计算与理想点的误差
            total_error += np.abs(rx_qam[sym_idx, k] - tx_qam[sym_idx, k])
            count += 1
    return total_error / count

error_no_pn = calculate_constellation_error(rx_qam_no_pn, qam_data, qam16_constellation)
error_pn = calculate_constellation_error(rx_qam_pn, qam_data, qam16_constellation)
error_pilot = calculate_constellation_error(rx_qam_pilot_corrected, qam_data, qam16_constellation)
error_df = calculate_constellation_error(rx_qam_df_corrected, qam_data, qam16_constellation)

print("相位噪声补偿效果对比")
print("-" * 50)
print(f"无相位噪声:        平均误差 = {error_no_pn:.6f}")
print(f"有相位噪声(未补偿): 平均误差 = {error_pn:.6f}")
print(f"导频辅助补偿:      平均误差 = {error_pilot:.6f}")
print(f"判决反馈跟踪:      平均误差 = {error_df:.6f}")
print("-" * 50)
print(f"导频辅助改善: {(error_pn - error_pilot) / error_pn * 100:.1f}%")
print(f"判决反馈改善: {(error_pn - error_df) / error_pn * 100:.1f}%")

In [ ]:
# 可视化补偿效果
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 选择第一个符号的子载波进行可视化
sym_idx = 10

# 1. 无相位噪声
ax1 = axes[0, 0]
tx_symbols = qam_data[sym_idx]
rx_symbols = rx_qam_no_pn[sym_idx]
ax1.scatter(tx_symbols.real, tx_symbols.imag, c='blue', s=50, alpha=0.5,
             label='发送', marker='x')
ax1.scatter(rx_symbols.real, rx_symbols.imag, c='green', s=50, alpha=0.5,
             label='接收')
ax1.axhline(y=0, color='k', linewidth=0.5)
ax1.axvline(x=0, color='k', linewidth=0.5)
ax1.set_xlabel('In-Phase (I)', fontsize=11)
ax1.set_ylabel('Quadrature (Q)', fontsize=11)
ax1.set_title('无相位噪声', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-1.5, 1.5)
ax1.set_ylim(-1.5, 1.5)
ax1.set_aspect('equal')

# 2. 有相位噪声未补偿
ax2 = axes[0, 1]
rx_symbols = rx_qam_pn[sym_idx]
ax2.scatter(tx_symbols.real, tx_symbols.imag, c='blue', s=50, alpha=0.5,
             label='发送', marker='x')
ax2.scatter(rx_symbols.real, rx_symbols.imag, c='red', s=50, alpha=0.5,
             label='接收')
ax2.axhline(y=0, color='k', linewidth=0.5)
ax2.axvline(x=0, color='k', linewidth=0.5)
ax2.set_xlabel('In-Phase (I)', fontsize=11)
ax2.set_ylabel('Quadrature (Q)', fontsize=11)
ax2.set_title(f'有相位噪声 (β={beta} rad²/s) - 未补偿', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-1.5, 1.5)
ax2.set_ylim(-1.5, 1.5)
ax2.set_aspect('equal')

# 3. 导频辅助补偿
ax3 = axes[1, 0]
rx_symbols = rx_qam_pilot_corrected[sym_idx]
ax3.scatter(tx_symbols.real, tx_symbols.imag, c='blue', s=50, alpha=0.5,
             label='发送', marker='x')
ax3.scatter(rx_symbols.real, rx_symbols.imag, c='purple', s=50, alpha=0.5,
             label='补偿后')
ax3.axhline(y=0, color='k', linewidth=0.5)
ax3.axvline(x=0, color='k', linewidth=0.5)
ax3.set_xlabel('In-Phase (I)', fontsize=11)
ax3.set_ylabel('Quadrature (Q)', fontsize=11)
ax3.set_title('导频辅助CPE补偿', fontsize=12)
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_xlim(-1.5, 1.5)
ax3.set_ylim(-1.5, 1.5)
ax3.set_aspect('equal')

# 4. 判决反馈跟踪
ax4 = axes[1, 1]
rx_symbols = rx_qam_df_corrected[sym_idx]
ax4.scatter(tx_symbols.real, tx_symbols.imag, c='blue', s=50, alpha=0.5,
             label='发送', marker='x')
ax4.scatter(rx_symbols.real, rx_symbols.imag, c='orange', s=50, alpha=0.5,
             label='补偿后')
ax4.axhline(y=0, color='k', linewidth=0.5)
ax4.axvline(x=0, color='k', linewidth=0.5)
ax4.set_xlabel('In-Phase (I)', fontsize=11)
ax4.set_ylabel('Quadrature (Q)', fontsize=11)
ax4.set_title('判决反馈相位跟踪', fontsize=12)
ax4.legend()
ax4.grid(True, alpha=0.3)
ax4.set_xlim(-1.5, 1.5)
ax4.set_ylim(-1.5, 1.5)
ax4.set_aspect('equal')

plt.tight_layout()
plt.show()

print("观察：")
print("1. 无相位噪声时，接收星座点与发送重合")
print("2. 有相位噪声时，星座点整体旋转")
print("3. 导频辅助补偿可以纠正整体旋转，但残余ICI仍存在")
print("4. 判决反馈可以更好跟踪相位变化，但可能引入误差传播")

## 8. 综合演示：完整相位噪声补偿系统

In [ ]:
# 完整相位噪声补偿系统演示

def simulate_complete_system(beta, snr_db, pilot_spacing=8):
    """
    模拟完整OFDM系统，包含相位噪声和补偿
    
    参数:
        beta: 相位噪声水平 (rad²/s)
        snr_db: 信噪比 (dB)
        pilot_spacing: 导频间隔
    
    返回:
        ber_no_pn: 无相位噪声时的BER
        ber_pn: 有相位噪声但未补偿的BER
        ber_corrected: 补偿后的BER
    """
    num_symbols = 100
    
    # 生成数据
    qam_data = np.random.choice(qam16_constellation, size=(num_symbols, N))
    
    # 调制
    tx_ofdm = ofdm_modulate(qam_data, N, cp_len)
    
    # 计算AWGN
    snr_linear = 10 ** (snr_db / 10)
    signal_power = np.mean(np.abs(tx_ofdm) ** 2)
    noise_power = signal_power / snr_linear
    
    # 导频索引
    pilot_indices = list(range(0, N, pilot_spacing))
    
    # 处理每个符号
    rx_ofdm_pn = []
    rx_ofdm_no_pn = []
    
    for sym_idx in range(num_symbols):
        total_samples = N + cp_len
        
        # 生成相位噪声
        phase_noise = generate_wiener_phase_noise(total_samples, dt, beta)
        
        # 生成AWGN
        noise = np.sqrt(noise_power / 2) * (np.random.randn(total_samples) + 1j * np.random.randn(total_samples))
        
        # 应用相位噪声和噪声
        tx_symbol = tx_ofdm[sym_idx]
        rx_symbol_pn = tx_symbol * np.exp(1j * phase_noise) + noise
        rx_symbol_no_pn = tx_symbol + noise
        
        rx_ofdm_pn.append(rx_symbol_pn)
        rx_ofdm_no_pn.append(rx_symbol_no_pn)
    
    rx_ofdm_pn = np.array(rx_ofdm_pn)
    rx_ofdm_no_pn = np.array(rx_ofdm_no_pn)
    
    # 解调
    rx_qam_pn = ofdm_demodulate(rx_ofdm_pn, N, cp_len)
    rx_qam_no_pn = ofdm_demodulate(rx_ofdm_no_pn, N, cp_len)
    
    # 补偿
    rx_qam_corrected = pilot_assisted_cpe_correction(rx_qam_pn, pilot_indices)
    
    # 计算BER
    def compute_ber(rx_qam, tx_qam, constellation):
        errors = 0
        total = 0
        for sym_idx in range(rx_qam.shape[0]):
            for k in range(rx_qam.shape[1]):
                distances = np.abs(constellation - rx_qam[sym_idx, k])
                rx_nearest = np.argmin(distances)
                distances_tx = np.abs(constellation - tx_qam[sym_idx, k])
                tx_nearest = np.argmin(distances_tx)
                if rx_nearest != tx_nearest:
                    errors += 1
                total += 1
        return errors / total
    
    ber_no_pn = compute_ber(rx_qam_no_pn, qam_data, qam16_constellation)
    ber_pn = compute_ber(rx_qam_pn, qam_data, qam16_constellation)
    ber_corrected = compute_ber(rx_qam_corrected, qam_data, qam16_constellation)
    
    return ber_no_pn, ber_pn, ber_corrected

# 测试不同SNR下的性能
snr_values = [10, 15, 20, 25, 30]
beta = 10.0  # rad²/s

print("SNR (dB) | 无PN BER | 有PN BER | 补偿后 BER")
print("-" * 55)

results = {'snr': [], 'no_pn': [], 'pn': [], 'corrected': []}

for snr in snr_values:
    ber_no_pn, ber_pn, ber_corrected = simulate_complete_system(beta, snr)
    print(f"  {snr:4d}   | {ber_no_pn:.6f} | {ber_pn:.6f} | {ber_corrected:.6f}")
    results['snr'].append(snr)
    results['no_pn'].append(ber_no_pn)
    results['pn'].append(ber_pn)
    results['corrected'].append(ber_corrected)

In [ ]:
# 绘制BER曲线
fig, ax = plt.subplots(figsize=(10, 6))

ax.semilogy(results['snr'], results['no_pn'], 'go-', linewidth=2,
            markersize=8, label='无相位噪声')
ax.semilogy(results['snr'], results['pn'], 'rx-', linewidth=2,
            markersize=8, label='有相位噪声（未补偿）')
ax.semilogy(results['snr'], results['corrected'], 'bs-', linewidth=2,
            markersize=8, label='导频辅助补偿')

ax.set_xlabel('SNR (dB)', fontsize=12)
ax.set_ylabel('误码率 (BER)', fontsize=12)
ax.set_title(f'OFDM系统BER vs SNR (相位噪声 β={beta} rad²/s)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim([1e-4, 1])

plt.tight_layout()
plt.show()

print("\n观察：")
print("1. 相位噪声在高SNR时对系统性能影响显著")
print("2. 导频辅助补偿可以显著降低BER")
print("3. 在低SNR时，相位噪声的影响被噪声淹没")
print("4. 补偿后BER曲线更接近无相位噪声的情况")

## 9. 思考题 (Review Questions)

1. **Wiener过程特性**：解释为什么Wiener过程适合建模相位噪声。它的哪个数学特性使其能够描述实际振荡器中的积分白噪声效应？

2. **CPE vs ICI**：在OFDM系统中，相位噪声引起的CPE和ICI有何本质区别？分别用什么方法补偿？为什么ICI更难处理？

3. **导频设计**：假设某OFDM系统有N=128个子载波，相位噪声的相干带宽为10个子载波宽度。请设计一个导频放置方案，使得能够在该相干带宽内准确估计CPE。

4. **高阶QAM敏感性**：64-QAM比16-QAM对相位噪声更敏感。定量分析：假设允许的相位误差为3°，64-QAM和16-QAM的最小Euclidean距离是多少？哪个更敏感？

5. **OTFS vs OFDM**：在相位噪声环境下，OTFS系统相比OFDM系统有何潜在优势？考虑延迟-多普勒域的稀疏性如何帮助处理相位噪声。

6. **实际振荡器规格**：某5G系统要求在5GHz载波频率下，相位噪声在1kHz偏移处不超过-100 dBc/Hz。计算该振荡器在1ms时间内的相位方差。这相当于多大的CPE（以弧度计）？

7. **补偿算法复杂度**：比较导频辅助补偿和判决反馈跟踪的计算复杂度。如果系统有N=1024个子载波，每个OFDM符号需要进行多少次复数乘法？

8. **CRLB分析**：推导在Wiener相位噪声模型下，CPE估计的克拉美-罗下限（Cramer-Rao Lower Bound）。提示：考虑相位噪声的先验分布和观测模型。

---

## 总结 (Summary)

本notebook介绍了相位噪声模型及其对OFDM和OTFS系统的影响：

- **相位噪声基础**：物理来源、频域表示（dBc/Hz）、对通信系统的影响

- **Wiener过程模型**：描述相位噪声的随机游走特性，1/f²功率谱密度，相位方差随时间线性增长

- **OFDM影响**：
  - CPE（公共相位旋转）：导致星座点整体旋转
  - ICI（子载波间干扰）：破坏子载波正交性
  - 高阶QAM更敏感

- **OTFS影响**：
  - 在延迟-多普勒域，信道稀疏性可用于处理
  - 相位噪声主要影响多普勒域

- **补偿方法**：
  - 导频辅助CPE估计
  - 判决反馈相位跟踪
  - 最大似然/贝叶斯估计

- **实际应用**：
  - 5G毫米波系统需要低相位噪声振荡器
  - 导频设计需要考虑相干带宽
  - 权衡开销与性能